# PyTorch on WebAssembly (Pyodide) — train a small MLP in the browser

This notebook runs entirely in your browser via a **Pyodide** kernel (no server).
It installs a from-source **CPU-only `torch` wheel built for `wasm32-emscripten`**
and trains a small multi-layer perceptron with **autograd + SGD**, showing the
training loss decreasing.

The wheel is a reduced build (single-threaded, no XNNPACK/MKLDNN/distributed/CUDA),
but eager autograd and `torch.nn` work. See the repo `RESULTS.md` for details.

In [ ]:
# Install the locally-shipped torch wheel into the Pyodide kernel.
import piplite, glob, js
# The wheel is shipped next to this notebook in the JupyterLite site.
await piplite.install("torch")  # resolved from the demo's piplite index
import torch
print("torch", torch.__version__)
print("default dtype", torch.get_default_dtype())

In [ ]:
# A tiny synthetic 2D binary-classification dataset (two Gaussian blobs).
import torch
torch.manual_seed(0)
N = 256
c0 = torch.randn(N, 2) * 0.6 + torch.tensor([-1.5, -1.5])
c1 = torch.randn(N, 2) * 0.6 + torch.tensor([ 1.5,  1.5])
X = torch.cat([c0, c1], dim=0)
y = torch.cat([torch.zeros(N), torch.ones(N)]).long()
print("X", tuple(X.shape), "y", tuple(y.shape))

In [ ]:
# Define a small MLP: Linear -> ReLU -> Linear.
import torch.nn as nn
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
print(model)

In [ ]:
# Train for a few epochs; record the loss curve.
losses = []
for epoch in range(60):
    opt.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()      # autograd
    opt.step()           # SGD update
    losses.append(float(loss))
    if epoch % 10 == 0 or epoch == 59:
        acc = (logits.argmax(1) == y).float().mean().item()
        print(f"epoch {epoch:3d}  loss={float(loss):.4f}  acc={acc:.3f}")
print("final loss", losses[-1])
assert losses[-1] < losses[0], "loss should decrease"

In [ ]:
# Plot the loss curve (matplotlib ships with Pyodide).
import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.plot(losses)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss")
plt.title("Training an MLP with PyTorch (wasm) in the browser")
plt.tight_layout(); plt.show()